# ASSIGNMENT 1: Graph Generation from an Architectural Model

## Overview
This assignment focuses on converting an architectural building model into a graph representation. By analyzing the spatial relationships between building components (rooms, doors, corridors), we create a network structure that captures both the geometric and topological properties of the architecture.

## Objectives
- Load and visualize a building model imported from Grasshopper
- Define nodes and edges based on architectural elements
- Generate a graph representation of spatial relationships
- Visualize the graph structure with nodes and edges
- Document the mapping logic between architectural elements and graph components

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph_ML-Giovanni_Carlo_Volpe\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [50]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.20) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [51]:
renderer = "vscode"

## 4. Load the Building Model (OBJ Files)

### Building Components
The model consists of three OBJ files representing:
- **Rooms**: The main spatial cells (rooms, spaces)
- **Doors**: Connection elements (apertures, transitions between rooms)
- **Exit**: Exit points and circulation spaces

These components will be combined to form a unified model for graph generation.

In [52]:
# Import all three OBJ files representing building components
objects_rooms = Topology.ByOBJPath(r"C:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph ML\Assignment_01_Giovanni-Carlo-Volpe\Assignment_01_Giovanni-Carlo-Volpe_rooms.obj")
objects_doors = Topology.ByOBJPath(r"C:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph ML\Assignment_01_Giovanni-Carlo-Volpe\Assignment_01_Giovanni-Carlo-Volpe_doors.obj")

print(objects_rooms)
print(objects_doors)
print("Building model loaded successfully")
print(f"  - Rooms: {len(objects_rooms)}")
print(f"  - Doors/Openings: {len(objects_doors)}")

[<topologic_core.Cluster object at 0x000001E44A4F4E30>, <topologic_core.Cluster object at 0x000001E44A4F4F30>, <topologic_core.Cluster object at 0x000001E45466C5F0>, <topologic_core.Cluster object at 0x000001E44A4EFB70>, <topologic_core.Cluster object at 0x000001E409FBB5B0>, <topologic_core.Cluster object at 0x000001E45466EDF0>, <topologic_core.Cluster object at 0x000001E45466D070>, <topologic_core.Cluster object at 0x000001E45466CA70>, <topologic_core.Cluster object at 0x000001E44BF3C9B0>, <topologic_core.Cluster object at 0x000001E448F9A370>, <topologic_core.Cluster object at 0x000001E45466CAF0>, <topologic_core.Cluster object at 0x000001E44BF3E630>, <topologic_core.Cluster object at 0x000001E44BF3DAB0>]
[<topologic_core.Cluster object at 0x000001E45683F2B0>, <topologic_core.Cluster object at 0x000001E45683C0F0>, <topologic_core.Cluster object at 0x000001E4546B3630>, <topologic_core.Cluster object at 0x000001E4546B3E70>, <topologic_core.Cluster object at 0x000001E4546B3BF0>, <topolog

### **Spatial Classification and Attribute Mapping**

This script categorizes building geometry by filtering volumetric cells based on their naming convention. 

* **Logic:** Identifies the **Lobby** and rooms belonging to **Wing A (WA)** and **Wing B (WB)**.
* **Processing:** Generates internal vertices, removes redundant collinear edges, and assigns visualization metadata.
* **Visuals:** Maps specific colors (**Red** for Lobby, **Green** for WA, **Blue** for WB) to the object dictionaries for clear spatial differentiation in the viewer.

In [53]:
cells = []
selectors = []

for object in objects_rooms:
    d = Topology.Dictionary(object)
    faces = Topology.Faces(object)
    
    # Process only objects with more than one face (volumetric cells)
    if len(faces) > 1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        
        # Extract the name property from the topology dictionary
        name = Dictionary.ValueAtKey(d, "name")
        
        # Assign colors based on room location/wing
        if "Lobby" in name:
            color = "red"
        elif "WA" in name: 
            # Logic for rooms in Wing A (room_WA_1 to room_WA_6)
            color = "green"
        elif "WB" in name: 
            # Logic for rooms in Wing B (room_WB_1 to room_WB_6)
            color = "blue"
        else:
            # Default color for any other objects
            color = "grey" 
            
        # Update dictionary with visualization attributes
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 20])
        s = Topology.SetDictionary(s, d)
        
        # Store results for downstream analysis or visualization
        selectors.append(s)
        cells.append(c)
        print(Dictionary.Keys(d), Dictionary.Values(d))

# Output the total count of processed cells
print(len(cells))

['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['grey', 'lobby', '', 'lobby', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_2', '', 'room_WB_2', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_5', '', 'room_WB_5', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_1', '', 'room_WB_1', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_3', '', 'room_WB_3', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_4', '', 'room_WB_4', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['blue', 'room_WB_6', '', 'room_WB_6', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['green', 'room_WA_2', '', 'room_WA_2', 1.0, 20]
['color', 'group', 'material', 'name', 'opacity', 'vertex_size'] ['green', 'room_WA_5', '', 'room_WA_5'

## 6. Show the geometry

In [42]:
Topology.Show(objects_rooms,
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

In [43]:
Topology.Show(objects_doors,
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7.Geometry to Cell

In [44]:
objects_rooms = Topology.ByOBJPath(r"C:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph ML\Assignment_01_Giovanni-Carlo-Volpe\Assignment_01_Giovanni-Carlo-Volpe_rooms.obj", selfMerge=True)
cells_rooms = Topology.Cells(objects_rooms[0])
cc_rooms = CellComplex.ByCells(cells_rooms)

Topology.Cells - Warning: The input is a Cell. Returning the same cell embedded in a list.
caller name: <module>


## 7.Show Cells

In [45]:
# Show geometry only (no vertices, no graph)
Topology.Show(cc_rooms,
              faceOpacity=0.1,
              faceColor="white",
              edgeColor="black",
              edgeWidth=1,
              showVertices=False,
              width=500,
              height=500,
              backgroundColor="white",
              renderer=renderer)


In [46]:
# Show geometry only (with vertices, no graph)
Topology.Show(cc_rooms,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 8. Derive primal graph

In [29]:
cluster = Cluster.ByTopologies([cc_rooms])
vertices = Topology.Vertices(cluster)
vertices = [Topology.Copy(v) for v in vertices]
edges = Topology.Edges(cluster)
edges = [Topology.Copy(e) for e in edges]
g1 = Graph.ByVerticesEdges(vertices, edges)

## 9. Assign visual attributes to the vertices and edges of the graph

In [30]:
vertices = Graph.Vertices(g1)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g1)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

## 10. Show the geometry and the graph

In [31]:
Topology.Show(cc_rooms, g1,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 11. Derive dual graph (based on cells in the cellcomplex)

In [32]:
g2 = Graph.ByTopology(cc_rooms, direct=True)

In [33]:
vertices = Graph.Vertices(g2)
for v in vertices:
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    v = Topology.SetDictionary(v, d)
edges = Graph.Edges(g2)
for e in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    e = Topology.SetDictionary(e, d)

In [34]:
Topology.Show(cc_rooms, g2,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.3,
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)